In [1]:
import pandas as pd
import sklearn as skl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay, classification_report
)
from sklearn.preprocessing import OneHotEncoder
import json
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

In [2]:
CSV_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
TARGET_COL = "Churn"
RANDOM_STATE = 0
TEST_SIZE = 0.25

# Load
df = pd.read_csv(CSV_PATH)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

mapping = {
    'No phone service': '0',
    'No internet service': '0',
    'No': '0',
    'Yes': '1'
}
df = df.replace(mapping)


# Split features / target
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Identify numeric and categorical columns (simple heuristic)
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.drop('customerID').tolist()

# Preprocessing pipelines
num_pipe = make_pipeline(
    SimpleImputer(strategy="mean"),
    StandardScaler()
)
cat_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False)
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Full pipelines (preprocessing + model)
logreg_pipeline = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
dummy_pipeline = make_pipeline(preprocessor, DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE))

# Fit
logreg_pipeline.fit(X_train, y_train)
dummy_pipeline.fit(X_train, y_train)

# Predict & evaluate
y_pred = logreg_pipeline.predict(X_test)
y_dummy = dummy_pipeline.predict(X_test)

# after your training and predictions (y_pred, y_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label="1", zero_division=0)
rec = recall_score(y_test, y_pred, pos_label="1", zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label="1", zero_division=0)
# try predict_proba for AUC if available
try:
    probs = logreg_pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test.map(int), probs)
except Exception:
    probs = None
    auc = None

with mlflow.start_run():
    mlflow.log_param("model_type", "LogisticRegression_pipeline")
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1", f1)
    if auc is not None:
        mlflow.log_metric("roc_auc", auc)

    # save transformed feature names (useful for coefficients)
    ohe = logreg_pipeline.named_steps["columntransformer"].named_transformers_["cat"].named_steps["onehotencoder"]
    cat_feature_names = []
    if hasattr(ohe, "get_feature_names_out"):
        cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
    feature_names = numeric_cols + cat_feature_names
    with open("../models/artifacts/feature_names.json", "w") as f:
        json.dump(feature_names, f)
    mlflow.log_artifact("../models/artifacts/feature_names.json")

    # log model
    mlflow.sklearn.log_model(logreg_pipeline, "logreg_pipeline")

2026/05/03 16:59:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 16:59:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
